# Base de données vols NYC 2021 : requêtes SQL pour Supabase

Requêtes utilisées pour construire la base dans Supabase (PostgreSQL), dans l'ordre d'exécution : création des tables, clés primaires, chargement et nettoyage des données, clés étrangères, sécurité.

| Table | Contenu | Clé primaire |
|---|---|---|
| `airlines` | compagnies | `carrier` |
| `airports` | aéroports | `faa` |
| `planes` | avions | `tailnum` |
| `weather` | météo horaire par aéroport | `origin, year, month, day, hour` |
| `flights` | vols (table centrale) | `year, month, day, hour, flight, carrier` |

**À savoir**
- Chaque cellule envoie sa requête à Supabase avec la fonction `executer` (étape 0). Les cellules de création modifient la base : ne lance pas le notebook d'un bloc sur une base déjà construite.
- Le chargement des CSV se fait hors notebook (import de l'éditeur de tables, ou `\copy` dans `psql`).
- Les fichiers sources notent les valeurs manquantes par un **espace seul** (`" "`), que les colonnes numériques refusent. L'étape 3 les détecte et les convertit en `NULL`.

## 0. Connexion

In [ ]:
# pip install sqlalchemy psycopg2-binary pandas
import getpass
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

# À adapter : Supabase > bouton "Connect" > "Session pooler" (ou "Direct connection")
HOTE = "aws-0-REGION.pooler.supabase.com"   # ou db.xxxxxxxx.supabase.co
UTILISATEUR = "postgres.xxxxxxxx"            # "postgres" en connexion directe
MOT_DE_PASSE = getpass.getpass("Mot de passe de la base : ")

engine = create_engine(URL.create(
    "postgresql+psycopg2",
    username=UTILISATEUR, password=MOT_DE_PASSE,
    host=HOTE, port=5432, database="postgres",
))


def executer(requete):
    """Exécute une ou plusieurs requêtes SQL. Renvoie un DataFrame si la dernière renvoie des lignes."""
    connexion = engine.raw_connection()
    try:
        curseur = connexion.cursor()
        curseur.execute(requete)
        resultat = None
        if curseur.description:
            colonnes = [c[0] for c in curseur.description]
            resultat = pd.DataFrame(curseur.fetchall(), columns=colonnes)
        connexion.commit()
        return resultat
    except Exception:
        connexion.rollback()
        raise
    finally:
        connexion.close()

## 1. Création des tables

Tables sans contrainte : les clés viennent à l'étape 2. La colonne d'index `Unnamed: 0` de `airports.xlsx` n'est pas reprise.

In [ ]:
executer("""
CREATE TABLE public.airlines (
    carrier  VARCHAR(2),
    name     VARCHAR(100)
);

CREATE TABLE public.airports (
    faa    VARCHAR(3),
    name   VARCHAR(100),
    lat    DOUBLE PRECISION,
    lon    DOUBLE PRECISION,
    alt    INTEGER,
    tz     SMALLINT,
    dst    CHAR(1),
    tzone  VARCHAR(50)
);

CREATE TABLE public.planes (
    tailnum       VARCHAR(10),
    year          SMALLINT,
    type          VARCHAR(50),
    manufacturer  VARCHAR(100),
    model         VARCHAR(50),
    engines       SMALLINT,
    seats         SMALLINT,
    speed         SMALLINT,
    engine        VARCHAR(30)
);

CREATE TABLE public.weather (
    origin      VARCHAR(3),
    year        SMALLINT,
    month       SMALLINT,
    day         SMALLINT,
    hour        SMALLINT,
    temp        REAL,
    dewp        REAL,
    humid       REAL,
    wind_dir    SMALLINT,
    wind_speed  REAL,
    wind_gust   REAL,
    precip      REAL,
    pressure    REAL,
    visib       REAL,
    time_hour   TIMESTAMPTZ
);

CREATE TABLE public.flights (
    year            SMALLINT,
    month           SMALLINT,
    day             SMALLINT,
    dep_time        SMALLINT,
    sched_dep_time  SMALLINT,
    dep_delay       SMALLINT,
    arr_time        SMALLINT,
    sched_arr_time  SMALLINT,
    arr_delay       SMALLINT,
    carrier         VARCHAR(2),
    flight          INTEGER,
    tailnum         VARCHAR(10),
    origin          VARCHAR(3),
    dest            VARCHAR(3),
    air_time        SMALLINT,
    distance        INTEGER,
    hour            SMALLINT,
    minute          SMALLINT,
    time_hour       TIMESTAMPTZ
);
""")

## 2. Clés primaires

La clé de `flights` inclut `carrier` : `(year, month, day, hour, flight)` n'est pas unique, car deux compagnies peuvent avoir le même numéro de vol à la même heure (2 389 doublons dans les données).

In [ ]:
executer("""
ALTER TABLE public.airlines ADD CONSTRAINT airlines_pk PRIMARY KEY (carrier);
ALTER TABLE public.airports ADD CONSTRAINT airports_pk PRIMARY KEY (faa);
ALTER TABLE public.planes   ADD CONSTRAINT planes_pk   PRIMARY KEY (tailnum);
ALTER TABLE public.weather  ADD CONSTRAINT weather_pk  PRIMARY KEY (origin, year, month, day, hour);
ALTER TABLE public.flights  ADD CONSTRAINT flights_pk  PRIMARY KEY (year, month, day, hour, flight, carrier);
""")

## 3. Chargement et nettoyage des données

### 3.1 Fichier `flights.csv`

Exporté depuis Excel, le fichier a chaque ligne entre guillemets (une seule colonne pour Postgres) et des espaces seuls pour les valeurs manquantes. Ce script retire les guillemets et vide ces cellules (elles sont ensuite importées comme `NULL`). Il remplace le fichier.

In [ ]:
from pathlib import Path

fichier = Path("flights.csv")
lignes = fichier.read_text(encoding="utf-8").replace('"', "").splitlines()
propres = [",".join(champ.strip() for champ in ligne.split(",")) for ligne in lignes]
fichier.write_text("\n".join(propres) + "\n", encoding="utf-8")

Les autres CSV (`airlines`, `airports`, `planes`) sont chargés avec l'import de Supabase.

### 3.2 Table `weather` : table de transit en texte

L'import direct dans `weather` échoue (`ERROR 22P02 : invalid input syntax for type real: " "`) à cause des espaces. On charge donc le CSV dans `weather_raw`, dont **toutes** les colonnes sont en `text`, puis on nettoie et on convertit.

In [ ]:
executer("""
CREATE TABLE public.weather_raw (
    origin text, year text, month text, day text, hour text,
    temp text, dewp text, humid text, wind_dir text, wind_speed text,
    wind_gust text, precip text, pressure text, visib text, time_hour text
);
""")

Importer `weather.csv` dans `weather_raw` (import CSV de Supabase), puis chercher les valeurs manquantes.

### 3.3 Détecter les valeurs `" "`

Lignes dont la pression est un espace seul :

In [ ]:
executer("""
SELECT *
FROM public.weather_raw AS w
WHERE w.pressure = ' '
LIMIT 100;
""")

Pour chercher dans **toutes** les colonnes à la fois, `WHERE w.* = ' '` n'est pas du SQL valide (on ne compare pas une ligne entière à une valeur). On transforme la ligne en JSON (`to_jsonb`) et on parcourt ses cellules. Cette requête renvoie les lignes qui ont au moins une cellule vide ou composée d'espaces :

In [ ]:
executer("""
SELECT w.*
FROM public.weather_raw AS w
WHERE EXISTS (
    SELECT 1
    FROM jsonb_each_text(to_jsonb(w)) AS c(colonne, valeur)
    WHERE TRIM(c.valeur) = ''
)
LIMIT 100;
""")

Nombre de cellules vides par colonne. Sur le fichier fourni : `wind_dir` 85, `wind_speed` 1, `wind_gust` 1, `pressure` 342.

In [ ]:
executer("""
SELECT c.colonne, COUNT(*) AS nb_vides
FROM public.weather_raw AS w,
     LATERAL jsonb_each_text(to_jsonb(w)) AS c(colonne, valeur)
WHERE TRIM(c.valeur) = ''
GROUP BY c.colonne
ORDER BY c.colonne;
""")

### 3.4 Nettoyer : espaces et cellules vides en `NULL`

`TRIM` retire les espaces et `NULLIF(..., '')` renvoie `NULL` quand il ne reste rien. Chaque colonne est traitée séparément : les valeurs correctes sont conservées.

In [ ]:
executer("""
UPDATE public.weather_raw
SET origin     = NULLIF(TRIM(origin), ''),
    wind_dir   = NULLIF(TRIM(wind_dir), ''),
    wind_speed = NULLIF(TRIM(wind_speed), ''),
    wind_gust  = NULLIF(TRIM(wind_gust), ''),
    pressure   = NULLIF(TRIM(pressure), ''),
""")

Contrôle : plus aucune ligne ne doit contenir d'espace seul.

In [ ]:
executer("""
SELECT *
FROM public.weather_raw AS w
WHERE w.wind_speed = ' '
LIMIT 100;
""")

In [ ]:
executer("""
SELECT c.colonne, COUNT(*) AS nb_vides
FROM public.weather_raw AS w,
     LATERAL jsonb_each_text(to_jsonb(w)) AS c(colonne, valeur)
WHERE TRIM(c.valeur) = ''
GROUP BY c.colonne;
""")

### 3.5 Convertir vers `weather` et supprimer la table de transit

In [ ]:
executer("""
INSERT INTO public.weather
SELECT
    origin,
    year::smallint,
    month::smallint,
    day::smallint,
    hour::smallint,
    temp::real,
    dewp::real,
    humid::real,
    wind_dir::smallint,
    wind_speed::real,
    wind_gust::real,
    precip::real,
    pressure::real,
    visib::real,
    time_hour::timestamptz
FROM public.weather_raw;
""")

Le fichier fourni contient 3 299 lignes (EWR uniquement, du 1er janvier au 18 mai 2021).

In [ ]:
executer("""
SELECT COUNT(*) AS nb_lignes FROM public.weather;
""")

In [ ]:
executer("""
DROP TABLE public.weather_raw;
""")

## 4. Compléter les tables de référence

Les clés étrangères vers `airports` et `planes` exigent que toutes les valeurs de `flights` y existent. Il manque 4 destinations et 716 immatriculations dans les fichiers sources.

### 4.1 Les 4 destinations absentes de `airports`

Ces 4 lignes ne sont pas dans les fichiers : leurs coordonnées et altitudes sont approximatives, à vérifier sur une source officielle (FAA, OurAirports) si la précision compte.

In [ ]:
executer("""
INSERT INTO public.airports (faa, name, lat, lon, alt, tz, dst, tzone) VALUES
('BQN', 'Rafael Hernandez Airport',               18.4949, -67.1294, 237, -4, 'N', 'America/Puerto_Rico'),
('PSE', 'Mercedita Airport',                      18.0083, -66.5630,  29, -4, 'N', 'America/Puerto_Rico'),
('SJU', 'Luis Munoz Marin International Airport', 18.4394, -66.0018,   9, -4, 'N', 'America/Puerto_Rico'),
('STT', 'Cyril E King Airport',                   18.3373, -64.9734,  23, -4, 'N', 'America/St_Thomas')
ON CONFLICT (faa) DO NOTHING;
""")

### 4.2 Les immatriculations absentes de `planes` (716)

Une ligne par immatriculation manquante, avec seulement `tailnum` renseigné. À lancer une fois `flights` chargée. Le `NOT EXISTS` est une anti-jointure : il ne garde que les `tailnum` de `flights` sans correspondance dans `planes`.

In [ ]:
executer("""
INSERT INTO public.planes (tailnum)
SELECT DISTINCT f.tailnum
FROM public.flights AS f
WHERE f.tailnum IS NOT NULL
  AND NOT EXISTS (
      SELECT 1 FROM public.planes AS p WHERE p.tailnum = f.tailnum
  );
""")

### 4.3 Contrôles : les deux résultats doivent valoir 0

In [ ]:
executer("""
SELECT 'destinations absentes de airports' AS controle, COUNT(DISTINCT f.dest) AS nb
FROM public.flights AS f
WHERE NOT EXISTS (SELECT 1 FROM public.airports a WHERE a.faa = f.dest)

UNION ALL
SELECT 'immatriculations absentes de planes', COUNT(DISTINCT f.tailnum)
FROM public.flights AS f
WHERE f.tailnum IS NOT NULL
  AND NOT EXISTS (SELECT 1 FROM public.planes p WHERE p.tailnum = f.tailnum);
""")

## 5. Clés étrangères

`flights` est la table centrale. Le lien avec `weather` est créé avec `NOT VALID` : la météo a des trous (heures manquantes, et le fichier fourni ne couvre qu'EWR jusqu'au 18 mai 2021), donc les lignes existantes ne sont pas vérifiées, afin de ne perdre aucun vol. Les nouvelles lignes restent contrôlées.

In [ ]:
executer("""
-- weather -> airports
ALTER TABLE public.weather
    ADD CONSTRAINT weather_origin_fk
    FOREIGN KEY (origin) REFERENCES public.airports (faa);

-- flights -> airlines
ALTER TABLE public.flights
    ADD CONSTRAINT flights_carrier_fk
    FOREIGN KEY (carrier) REFERENCES public.airlines (carrier);

-- flights -> airports (départ et arrivée)
ALTER TABLE public.flights
    ADD CONSTRAINT flights_origin_fk
    FOREIGN KEY (origin) REFERENCES public.airports (faa);

ALTER TABLE public.flights
    ADD CONSTRAINT flights_dest_fk
    FOREIGN KEY (dest) REFERENCES public.airports (faa);

-- flights -> planes
ALTER TABLE public.flights
    ADD CONSTRAINT flights_tailnum_fk
    FOREIGN KEY (tailnum) REFERENCES public.planes (tailnum);

-- flights -> weather (clé composite, lignes existantes non vérifiées)
ALTER TABLE public.flights
    ADD CONSTRAINT flights_weather_fk
    FOREIGN KEY (origin, year, month, day, hour)
    REFERENCES public.weather (origin, year, month, day, hour)
    NOT VALID;
""")

Index sur les colonnes de `flights` qui portent les liens (Postgres ne les crée pas automatiquement) :

In [ ]:
executer("""
CREATE INDEX IF NOT EXISTS idx_flights_carrier ON public.flights (carrier);
CREATE INDEX IF NOT EXISTS idx_flights_origin  ON public.flights (origin);
CREATE INDEX IF NOT EXISTS idx_flights_dest    ON public.flights (dest);
CREATE INDEX IF NOT EXISTS idx_flights_tailnum ON public.flights (tailnum);
CREATE INDEX IF NOT EXISTS idx_flights_weather ON public.flights (origin, year, month, day, hour);
""")

## 6. Sécurité (RLS)

Chaque table a la sécurité par ligne activée et une seule règle : lecture ouverte à tous (`SELECT`), aucune règle d'écriture. Toute personne qui possède la clé publique du projet peut donc **lire** les tables par l'API, mais pas les modifier.

In [ ]:
executer("""
ALTER TABLE public.airlines ENABLE ROW LEVEL SECURITY;
ALTER TABLE public.airports ENABLE ROW LEVEL SECURITY;
ALTER TABLE public.planes   ENABLE ROW LEVEL SECURITY;
ALTER TABLE public.weather  ENABLE ROW LEVEL SECURITY;
ALTER TABLE public.flights  ENABLE ROW LEVEL SECURITY;

CREATE POLICY "lecture publique airlines" ON public.airlines FOR SELECT USING (true);
CREATE POLICY "lecture publique airports" ON public.airports FOR SELECT USING (true);
CREATE POLICY "lecture publique planes"   ON public.planes   FOR SELECT USING (true);
CREATE POLICY "lecture publique weather"  ON public.weather  FOR SELECT USING (true);
CREATE POLICY "lecture publique flights"  ON public.flights  FOR SELECT USING (true);
""")

## 7. Vérifications

In [ ]:
executer("""
SELECT conname, contype, conrelid::regclass AS table_source, confrelid::regclass AS table_cible, convalidated
FROM pg_constraint
WHERE connamespace = 'public'::regnamespace
ORDER BY contype, conname;
""")

Attendu : 5 clés primaires (`p`) et 6 clés étrangères (`f`), dont `flights_weather_fk` avec `convalidated = false` (créée avec `NOT VALID`).

Indicateurs :

In [ ]:
executer("""
SELECT 'Nombre total d''aéroports' AS indicateur, COUNT(*) AS valeur
FROM public.airports

UNION ALL
SELECT 'Nombre d''aéroports de départ', COUNT(DISTINCT origin)
FROM public.flights

UNION ALL
SELECT 'Nombre d''aéroports de destination', COUNT(DISTINCT dest)
FROM public.flights

UNION ALL
SELECT 'Aéroports sans heure d''été', COUNT(*)
FROM public.airports
WHERE dst = 'N'

UNION ALL
SELECT 'Nombre de valeurs dans tzone', COUNT(DISTINCT tzone)
FROM public.airports

UNION ALL
SELECT 'Nombre de compagnies', COUNT(*)
FROM public.airlines

UNION ALL
SELECT 'Nombre d''avions', COUNT(*)
FROM public.planes

UNION ALL
SELECT 'Nombre de vols annulés', COUNT(*)
FROM public.flights
WHERE dep_time IS NULL;
""")

## 8. Exemples de requêtes

### 8.1 Jointure : vols par aéroport de départ

In [ ]:
executer("""
SELECT
    f.origin,
    a.name,
    COUNT(*) AS nb_vols
FROM public.flights AS f
JOIN public.airports AS a ON a.faa = f.origin
GROUP BY f.origin, a.name
ORDER BY nb_vols DESC;
""")

### 8.2 Jointure externe avec la météo

`LEFT JOIN` conserve les vols sans mesure météo (colonnes météo à `NULL`).

In [ ]:
executer("""
SELECT f.origin, f.year, f.month, f.day, f.hour, f.dep_delay, w.temp, w.wind_speed
FROM public.flights AS f
LEFT JOIN public.weather AS w
  ON  w.origin = f.origin
  AND w.year   = f.year
  AND w.month  = f.month
  AND w.day    = f.day
  AND w.hour   = f.hour
LIMIT 100;
""")

### 8.3 Anti-jointure : vols sans mesure météo

`NOT EXISTS` ne garde que les vols pour lesquels aucune ligne de `weather` ne correspond.

In [ ]:
executer("""
SELECT f.origin, COUNT(*) AS vols_sans_meteo
FROM public.flights AS f
WHERE NOT EXISTS (
    SELECT 1
    FROM public.weather AS w
    WHERE w.origin = f.origin
      AND w.year   = f.year
      AND w.month  = f.month
      AND w.day    = f.day
      AND w.hour   = f.hour
)
GROUP BY f.origin;
""")

## 9. Optionnel : exécuter du SQL depuis `supabase-py`

Fonction `exec_sql`, appelable avec `supabase.rpc("exec_sql", {"query": "SELECT ..."})`. Elle est réservée au rôle `service_role` (clé secrète `sb_secret_...`). Avec la clé publique, l'appel échoue (`permission denied for function exec_sql`), ce qui est voulu : ne l'ouvre pas à `anon`.

In [ ]:
executer("""
CREATE OR REPLACE FUNCTION public.exec_sql(query text)
RETURNS jsonb
LANGUAGE plpgsql
SECURITY DEFINER
SET search_path = public
AS $$
DECLARE
    resultat jsonb;
BEGIN
    EXECUTE format(
        'SELECT COALESCE(jsonb_agg(t), ''[]''::jsonb) FROM (%s) t', query
    ) INTO resultat;
    RETURN resultat;
END;
$$;

REVOKE ALL ON FUNCTION public.exec_sql(text) FROM PUBLIC, anon, authenticated;
GRANT EXECUTE ON FUNCTION public.exec_sql(text) TO service_role;
""")

In [ ]:
# pip install supabase
import os
from supabase import create_client

supabase = create_client("https://xxxxxxxx.supabase.co", os.environ["SUPABASE_SECRET_KEY"])

requete = """
SELECT f.origin, a.name, COUNT(*) AS nb_vols
FROM public.flights AS f
JOIN public.airports AS a ON a.faa = f.origin
GROUP BY f.origin, a.name
ORDER BY nb_vols DESC
"""

reponse = supabase.rpc("exec_sql", {"query": requete.strip().rstrip(";")}).execute()
pd.DataFrame(reponse.data)